In [1]:
import sys
print(sys.executable)

/home/hollowedblood/.conda/envs/senior-project/bin/python


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import shared_data

In [3]:
X_train, y_train, X_test, y_test = shared_data.get_xyts(
    condition="noise",
    test_size=0.2,
    lookback=63,
    horizon=5,
    deadzone=0.005,
    scale=True,
)

In [4]:
X_train.shape

torch.Size([2186, 63, 20, 13])

In [5]:
y_train.shape

torch.Size([2186, 20])

In [6]:
X_test.shape

torch.Size([548, 63, 20, 13])

In [7]:
y_test.shape

torch.Size([548, 20])

In [8]:
train_dataset = TensorDataset(X_train, y_train)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

In [10]:
X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape)
print("y:", y_batch.shape)

X: torch.Size([32, 63, 20, 13])
y: torch.Size([32, 20])


In [11]:
class Baugus_CNNLSTM_Net(nn.Module):
    def __init__(self):
        super(Baugus_CNNLSTM_Net,self).__init__()

        self.cnn1 = nn.Conv1d(
            # Output shape: [batch, out_channels, lookback - kernel_size + 1]
            # Assumes stride=1, padding=0, dilation=1.
            in_channels=X_train.shape[2] * X_train.shape[3], # Tickers × features per ticker
            out_channels=128, # Number of learned filters; each produces one output channel
            kernel_size=3, # Consecutive trading days examined at each position
        )
        self.lstm1 = nn.LSTM(
            input_size=self.cnn1.out_channels, # Number of CNN features at each time position
            hidden_size=64, # Size of the learned hidden state
            num_layers=1, # One recurrent layer / parameter
            batch_first=True, # Input order: [batch, time, features]
        )
        self.fc1 = nn.Linear(
            in_features=self.lstm1.hidden_size, # decided by LSTM hidden_size
            out_features=X_train.shape[2] * 3, # Three class scores for each ticker
        )

    def forward(self, x):
        x = x.flatten(start_dim=2) #combine ticker and features
        x = x.transpose(1,2) # flip to [batch, channels, time]
        x = F.relu(self.cnn1(x)) # run through cnn1
        
        x = x.transpose(1,2) # flip to [batch, time, cnn features]
        x, _ = self.lstm1(x) # run through lstm
        x = x[:,-1,:] # select the last time position
        
        x = self.fc1(x) # run through linear layer

        return x.reshape(x.shape[0], -1, 3) # shape to [batch, ticker, class]

In [12]:
model = Baugus_CNNLSTM_Net()

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [15]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = model(x_batch)
        loss = criterion(predictions.transpose(1, 2), y_batch) #convert [32, 20, 3] > [32, 3, 20]

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}: Loss was {running_loss / len(train_loader)}")

Epoch 1: Loss was 1.0004208087921143
Epoch 2: Loss was 0.9923228621482849
Epoch 3: Loss was 0.9775397181510925
Epoch 4: Loss was 0.9696559309959412
Epoch 5: Loss was 0.9618602991104126
Epoch 6: Loss was 0.9505866169929504
Epoch 7: Loss was 0.9389641880989075
Epoch 8: Loss was 0.9300927519798279
Epoch 9: Loss was 0.9220072627067566
Epoch 10: Loss was 0.9264827966690063
Epoch 11: Loss was 0.9122378826141357
Epoch 12: Loss was 0.9021693468093872
Epoch 13: Loss was 0.8894304037094116
Epoch 14: Loss was 0.8835538625717163
Epoch 15: Loss was 0.882931113243103
Epoch 16: Loss was 0.8673701882362366
Epoch 17: Loss was 0.8561404347419739
Epoch 18: Loss was 0.8817189931869507
Epoch 19: Loss was 0.8808361291885376
Epoch 20: Loss was 0.8696598410606384


In [18]:
with torch.no_grad():
    model.eval()

    predictions = model(X_test)
    loss = criterion(predictions.transpose(1, 2), y_test)

    predicted_classes = predictions.argmax(dim=-1)  # [samples, 20]
    accuracy = (predicted_classes == y_test).float().mean().item()

    print(f"Test loss: {loss.item():.4f}")
    print(f"Test accuracy: {accuracy:.2%}")

majority_classes = y_train.mode(dim=0).values
baseline_accuracy = (y_test == majority_classes).float().mean().item()
print(f"Baseline accuracy: {baseline_accuracy:.2%}")

Test loss: 1.0067
Test accuracy: 45.92%
Baseline accuracy: 47.35%
